# AI Job Agent — PHASE 1: Data Preparation

**الهدف:** بناء corpus وظائف نظيف ومُهيكل، **مستقل تمامًا عن أي مستخدم أو استعلام**.

هذا الملف لا يعرف شيئًا عن السيرة الذاتية ولا عن تفضيلات المستخدم. مخرجاته تُستهلك لاحقًا في:

| المخرج | يُستخدم في |
|---|---|
| `jobs_prepared.parquet` | Phase 3 / 4 / 5 (الفلترة والمطابقة) |
| `job_embeddings.npy` + `job_ids.npy` | Phase 3 (Embedding Retrieval) |
| `skill_vocabulary.parquet` | Phase 5 (مطابقة المهارات) |
| `DATA_DICTIONARY.md` | توثيق الـ schema |

**Pipeline:**

```
Multi-anchor fetch  →  Normalize text  →  Language / Spam filters
      →  Exact + Near duplicate removal  →  Location & date normalization
      →  Quality gate  →  LLM structured extraction (async + cache)
      →  Skill normalization  →  QA  →  jobs_prepared.parquet
```

---
## 1. Setup

In [ ]:
# Clone the Open Jobs repository
!git clone -q https://github.com/elliottdehn/open-jobs.git 2>/dev/null || echo "already cloned"
%cd /content/open-jobs

!pip install -q uv openai pydantic tqdm nest_asyncio

In [ ]:
import os
import re
import json
import html
import time
import math
import base64
import hashlib
import asyncio
import unicodedata
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 120)

WORK = Path("work")
CACHE = Path("cache")
OUT = Path("outputs")

for d in (WORK, CACHE, OUT):
    d.mkdir(parents=True, exist_ok=True)

print("Ready.")

---
## 2. Configuration

كل قرارات الـ pipeline مجمّعة هنا. لا توجد قيم مبعثرة داخل الخلايا.

In [ ]:
CONFIG = {

    # ---------- Corpus scope ----------
    # نقاط ارتكاز عريضة لبناء corpus متنوع.
    # هذه ليست تفضيلات مستخدم — هي فقط طريقة open-jobs
    # لاسترجاع مجموعات وظائف من الأرشيف.
    # قُلّصت من 6 نقاط إلى 3 لتقليل حجم الـ corpus (وبالتالي التوكنز
    # المستهلكة في الاستخراج). 3 نقاط عريضة تكفي لتغطية متنوعة كافية
    # لمشروع تخرّج. أضف نقاطًا أخرى لاحقًا فقط إذا احتجت مجالات إضافية.
    "anchors": [
        {
            "name": "data_tech",
            "title": "Data Analyst",
            "location": "Saudi Arabia",
            "text": "Data analysis, business intelligence, SQL, Python, "
                    "dashboards, reporting, data engineering, machine learning, "
                    "software development, backend, frontend, APIs, cloud.",
        },
        {
            "name": "business_ops",
            "title": "Business Analyst",
            "location": "Saudi Arabia",
            "text": "Business analysis, project management, operations, "
                    "strategy, consulting, finance, accounting, human resources, "
                    "recruitment, administration.",
        },
        {
            "name": "engineering_other",
            "title": "Engineer",
            "location": "Saudi Arabia",
            "text": "Mechanical, electrical, civil, industrial engineering, "
                    "maintenance, construction, QA/QC, nursing, medical, "
                    "sales, marketing, customer service.",
        },
    ],

    # عدد المجموعات المسترجعة لكل نقطة ارتكاز — قُلّص من 20 إلى 8.
    # 3 anchors × 8 groups يعطي corpus بحجم ~4-6K وظيفة بعد التنظيف،
    # كافٍ لعرض retrieval/ranking/evaluation بدون تضخيم فاتورة الاستخراج.
    "fetch_top_groups": 8,

    # ---------- Cleaning thresholds ----------
    "min_jd_chars": 250,
    "max_jd_chars": 25_000,
    "min_title_chars": 3,
    "near_dup_cosine": 0.985,

    # اللغات المقبولة في الـ corpus. None = اقبل الكل
    "allowed_languages": ["en"],

    # احذف الوظائف الأقدم من هذا العمر بالأيام. None = بدون فلترة
    "max_age_days": None,

    # ---------- LLM extraction ----------
    "extraction_model": "gpt-5.6-luna",
    "prompt_version": "v2.0",

    # حد أقصى للوظائف المُستخرجة (للتحكم بالتكلفة). None = الكل
    "max_jobs_to_extract": None,

    # قُلّص من 12,000 إلى 6,000 حرف (~1,500 توكن). الثلث الأخير من أغلب
    # الوصف الوظيفي غالبًا EEO/بيانات شركة/تعليمات تقديم — لا مهارات فيه.
    "max_jd_chars_sent_to_llm": 6_000,
    "concurrency": 8,
    "max_retries": 5,
    "checkpoint_every": 200,

    # ---------- GitHub auto-upload (Section 18) ----------
    # عدّل هذين السطرين بقيمك، والتوكن يُقرأ من Colab Secrets (لا يُكتب هنا مطلقًا)
    "github_repo": "USERNAME/REPO_NAME",   # مثال: "khaled/ai-job-agent-data"
    "github_branch": "main",
    "github_subdir": "phase1_outputs",     # المجلد داخل الـ repo الذي تُنسخ له المخرجات
}

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
print("RUN_ID:", RUN_ID)

---
## 3. Build the raw corpus (multi-anchor)

أداة `open-jobs` تسترجع مجموعات وظائف قريبة من وصف مرجعي. نقاط ارتكاز متعددة
تعطي تغطية أوسع من نقطة واحدة، وبالتالي corpus قابل لخدمة أي مستخدم لاحقًا.

> عمود `sim` ناتج عن نقطة الارتكاز وليس خاصية للوظيفة — سيُحذف بعد الدمج.

In [ ]:
def fetch_anchor(anchor: dict, top_groups: int) -> pd.DataFrame:
    """Run one open-jobs embed+fetch cycle and return the raw dataframe."""

    anchor_path = WORK / f"anchor_{anchor['name']}.md"
    anchor_path.write_text(
        f"Title: {anchor['title']}\n\n"
        f"Location: {anchor['location']}\n\n"
        f"{anchor['text']}\n",
        encoding="utf-8",
    )

    embed_cmd = (
        f'uv run tools/jobs.py embed '
        f'--file "{anchor_path}" '
        f'--title "{anchor["title"]}" '
        f'--location "{anchor["location"]}"'
    )
    fetch_cmd = f"uv run tools/jobs.py fetch --top {top_groups}"

    rc_embed = os.system(embed_cmd)
    rc_fetch = os.system(fetch_cmd)

    if rc_embed != 0 or rc_fetch != 0:
        print(f"  [WARN] anchor '{anchor['name']}' failed "
              f"(embed={rc_embed}, fetch={rc_fetch})")
        return pd.DataFrame()

    part = pd.read_parquet(WORK / "jobs.parquet")
    part["anchor"] = anchor["name"]

    # احفظ نسخة لكل نقطة ارتكاز حتى لا نعيد التحميل عند إعادة التشغيل
    part.to_parquet(WORK / f"raw_{anchor['name']}.parquet", index=False)
    return part

In [ ]:
frames = []

for anchor in CONFIG["anchors"]:
    cached = WORK / f"raw_{anchor['name']}.parquet"

    if cached.exists():
        part = pd.read_parquet(cached)
        print(f"{anchor['name']:<18} cached   {len(part):>7,} jobs")
    else:
        part = fetch_anchor(anchor, CONFIG["fetch_top_groups"])
        print(f"{anchor['name']:<18} fetched  {len(part):>7,} jobs")

    if len(part):
        frames.append(part)

raw_df = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows before merge-dedup: {len(raw_df):,}")

In [ ]:
# صف واحد لكل وظيفة فريدة عبر نقاط الارتكاز
raw_df = (
    raw_df
    .sort_values("sim", ascending=False)
    .drop_duplicates(subset=["ats", "slug", "id"], keep="first")
    .reset_index(drop=True)
)

# sim خاصية للاستعلام لا للوظيفة — لا مكان لها في corpus مُجهّز
raw_df = raw_df.drop(columns=["sim"], errors="ignore")

print(f"Unique jobs: {len(raw_df):,}")
print(f"Columns: {raw_df.columns.tolist()}")
raw_df.head(3)

---
## 4. Stable job identifier

مفتاح ثابت لا يتغير بين التشغيلات — ضروري للـ dedup، والـ cache،
وربط نتائج التقييم في Phase 9.

In [ ]:
def make_job_id(row) -> str:
    key = f"{row['ats']}|{row['slug']}|{row['id']}"
    return hashlib.sha1(key.encode("utf-8")).hexdigest()[:16]


raw_df["job_id"] = raw_df.apply(make_job_id, axis=1)

assert raw_df["job_id"].is_unique, "job_id collision detected"
print(f"job_id generated for {len(raw_df):,} jobs")
raw_df[["job_id", "title", "company"]].head()

---
## 5. Text normalization

ثلاث نسخ من كل نص، لكل منها غرض مختلف:

| العمود | المعالجة | الاستخدام |
|---|---|---|
| `jd_clean` | HTML مفكوك، الأسطر محفوظة | LLM extraction، العرض للمستخدم |
| `jd_flat` | سطر واحد | التشابه النصي، الفهرسة |
| `jd_raw` | الأصل | التدقيق والمراجعة |

> الخطأ الشائع: تطبيق `\s+ → " "` على كل شيء. هذا يدمّر بنية القوائم
> والعناوين داخل الوصف الوظيفي، ويضعف جودة الاستخراج.

In [ ]:
TAG_RE = re.compile(r"<[^>]{1,400}?>")
SCRIPT_RE = re.compile(r"<(script|style)[^>]*>.*?</\1>", re.S | re.I)
BULLET_RE = re.compile(r"^[\s\u2022\u25cf\u25aa\u00b7\-\*\u2013\u2014]+", re.M)
INLINE_WS_RE = re.compile(r"[ \t\x0b\f\r\u00a0\u2000-\u200a]+")
MANY_NL_RE = re.compile(r"\n{3,}")
ZERO_WIDTH_RE = re.compile(r"[\u200b-\u200f\ufeff\u2060]")


def clean_text(value, keep_newlines: bool = True) -> str:
    """Normalize HTML-ish job text while preserving useful structure."""

    if not isinstance(value, str) or not value.strip():
        return ""

    text = value

    # كيانات HTML قد تكون مُرمّزة مرتين (&amp;amp;)
    for _ in range(3):
        new = html.unescape(text)
        if new == text:
            break
        text = new

    text = SCRIPT_RE.sub(" ", text)
    text = TAG_RE.sub("\n", text)
    text = ZERO_WIDTH_RE.sub("", text)
    text = unicodedata.normalize("NFKC", text)

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = INLINE_WS_RE.sub(" ", text)

    lines = [ln.strip() for ln in text.split("\n")]
    lines = [ln for ln in lines if ln]
    text = "\n".join(lines)
    text = MANY_NL_RE.sub("\n\n", text)

    if not keep_newlines:
        text = text.replace("\n", " ")
        text = INLINE_WS_RE.sub(" ", text)

    return text.strip()


def norm_key(value) -> str:
    """Aggressive normalization used only for matching and dedup keys."""

    text = clean_text(value, keep_newlines=False).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
df = raw_df.copy()

df["jd_raw"] = df["jd"]

df["title_clean"] = df["title"].map(lambda x: clean_text(x, keep_newlines=False))
df["company_clean"] = df["company"].map(lambda x: clean_text(x, keep_newlines=False))
df["location_raw"] = df["location"].map(lambda x: clean_text(x, keep_newlines=False))
df["url_clean"] = df["url"].fillna("").astype(str).str.strip()

df["jd_clean"] = df["jd"].map(lambda x: clean_text(x, keep_newlines=True))
df["jd_flat"] = df["jd_clean"].map(lambda x: clean_text(x, keep_newlines=False))

df["title_norm"] = df["title_clean"].map(norm_key)
df["company_norm"] = df["company_clean"].map(norm_key)

df["jd_chars"] = df["jd_clean"].str.len()
df["jd_words"] = df["jd_flat"].str.split().map(len)

print(f"Rows: {len(df):,}")
df[["title_clean", "company_clean", "jd_chars", "jd_words"]].describe(include="all").head(4)

In [ ]:
# تحقّق: لم تعد كيانات HTML موجودة
leftover = df["jd_clean"].str.contains(r"&(amp|lt|gt|nbsp|#\d+);", regex=True, na=False).sum()
tags_left = df["jd_clean"].str.contains(r"<[a-zA-Z/][^>]*>", regex=True, na=False).sum()

print("Rows with leftover HTML entities:", leftover)
print("Rows with leftover HTML tags   :", tags_left)

print("\n--- sample ---")
print(df["jd_clean"].iloc[0][:600])

---
## 6. Language detection

الأرشيف يحتوي إعلانات عربية وإنجليزية ومختلطة. بدون تمييز اللغة،
الـ embeddings تختلط والـ extraction يتشوّش. الاعتماد على نسبة الحروف
يكفي هنا ولا يحتاج مكتبة خارجية.

In [ ]:
AR_CHARS = re.compile(r"[\u0600-\u06FF\u0750-\u077F]")
LAT_CHARS = re.compile(r"[A-Za-z]")


def detect_lang(text: str) -> str:
    if not text:
        return "unknown"

    n_ar = len(AR_CHARS.findall(text))
    n_lat = len(LAT_CHARS.findall(text))
    total = n_ar + n_lat

    if total < 30:
        return "unknown"

    ar_ratio = n_ar / total

    if ar_ratio >= 0.60:
        return "ar"
    if ar_ratio <= 0.10:
        return "en"
    return "mixed"


df["jd_lang"] = df["jd_flat"].map(detect_lang)
print(df["jd_lang"].value_counts())

---
## 7. Spam and aggregator filtering

الأرشيف يحوي إعلانات مجمّعة من مواقع وسيطة: عنوان واحد يغطي عشرات الوظائف،
أو صفحة تسويقية بلا محتوى وظيفي. هذه تلوّث الـ retrieval بشدة لأنها تتطابق
سطحيًا مع أي استعلام.

نضع علامة أولًا ثم نفلتر — حتى نتمكن من مراجعة ما حُذف.

In [ ]:
SPAM_TITLE_PATTERNS = [
    r"\bmultiple jobs\b",
    r"\bjobs? (in|for) .{0,40}\b20\d\d\b",
    r"\bapply now\b",
    r"\bverified overseas\b",
    r"\bwalk[- ]?in\b",
    r"\burgent(ly)? (hiring|required)\b",
    r"\bjob vacancies?\b",
    r"\bfree recruitment\b",
    r"\bvisa (sponsorship )?available\b",
    r"\bhiring now\b.*\bjobs\b",
    r"\b\d{2,} (job )?(vacancies|openings|positions)\b",
]

SPAM_JD_PATTERNS = [
    r"\bwhatsapp\b.{0,30}\+?\d[\d\s\-]{7,}",
    r"\bsend (your )?cv (to|on) .{0,40}@",
    r"\bregistration fee\b",
    r"\bno (registration )?(fee|charges)\b.{0,40}\bagent\b",
]

SPAM_TITLE_RE = re.compile("|".join(SPAM_TITLE_PATTERNS), re.I)
SPAM_JD_RE = re.compile("|".join(SPAM_JD_PATTERNS), re.I)


def count_title_like_lines(jd: str) -> int:
    """Aggregator pages list many roles as bare short lines."""

    lines = [ln.strip() for ln in jd.split("\n") if ln.strip()]
    short = [
        ln for ln in lines
        if 8 <= len(ln) <= 45 and not ln.endswith((".", ":", ";"))
    ]
    return len(short)


df["spam_title_hit"] = df["title_clean"].str.contains(SPAM_TITLE_RE, na=False)
df["spam_jd_hit"] = df["jd_flat"].str.contains(SPAM_JD_RE, na=False)
df["title_like_lines"] = df["jd_clean"].map(count_title_like_lines)

df["is_spam"] = (
    df["spam_title_hit"]
    | df["spam_jd_hit"]
    | (df["title_like_lines"] >= 25)
)

print(f"Flagged as spam: {df['is_spam'].sum():,} / {len(df):,}")
df.loc[df["is_spam"], ["title_clean", "company_clean", "title_like_lines"]].head(15)

In [ ]:
# مراجعة بشرية سريعة قبل الحذف — تجنّبًا لحذف وظائف صحيحة
sample_spam = df.loc[df["is_spam"]].sample(
    n=min(5, int(df["is_spam"].sum())),
    random_state=42,
) if df["is_spam"].any() else df.head(0)

for _, row in sample_spam.iterrows():
    print("=" * 90)
    print("TITLE  :", row["title_clean"])
    print("COMPANY:", row["company_clean"])
    print("REASON :",
          "title" if row["spam_title_hit"] else "",
          "jd" if row["spam_jd_hit"] else "",
          f"lines={row['title_like_lines']}")
    print(row["jd_clean"][:400])

---
## 8. Duplicate removal

ثلاث طبقات، من الأرخص إلى الأغلى:

1. **URL متطابق** — نفس الإعلان مُسترجع مرتين
2. **بصمة المحتوى** — `sha1(title_norm + jd_norm)`
3. **شبه مكرر** — نفس الوظيفة عبر وكالات مختلفة، تُكشف بتشابه
   الـ embedding داخل مجموعات العنوان الواحد

الطبقة الثالثة محصورة داخل مجموعات العنوان المطبّع، فتتجنب
حساب مصفوفة `n²` كاملة.

In [ ]:
before = len(df)

# ---- الطبقة 1: URL ----
mask_url = df["url_clean"].ne("")
df = pd.concat([
    df[~mask_url],
    df[mask_url].drop_duplicates(subset=["url_clean"], keep="first"),
]).sort_index()

print(f"After URL dedup      : {len(df):,}  (-{before - len(df):,})")

# ---- الطبقة 2: بصمة المحتوى ----
before = len(df)
df["content_hash"] = (
    df["title_norm"] + "||" + df["jd_flat"].map(norm_key)
).map(lambda s: hashlib.sha1(s.encode("utf-8")).hexdigest())

df = df.drop_duplicates(subset=["content_hash"], keep="first")
print(f"After content dedup  : {len(df):,}  (-{before - len(df):,})")

df = df.reset_index(drop=True)

In [ ]:
def decode_vec(b64: str, dim: int = 1536) -> np.ndarray:
    """open-jobs stores float32 little-endian vectors as base64."""

    if not isinstance(b64, str) or not b64:
        return np.zeros(dim, dtype=np.float32)

    raw = base64.b64decode(b64)
    vec = np.frombuffer(raw, dtype="<f4")
    return vec.astype(np.float32)


vectors = np.vstack(df["vec_b64"].map(decode_vec).values)

norms = np.linalg.norm(vectors, axis=1, keepdims=True)
norms[norms == 0] = 1.0
vectors_unit = vectors / norms

print("Embedding matrix:", vectors.shape)

In [ ]:
def near_duplicate_ids(frame: pd.DataFrame,
                       unit_vectors: np.ndarray,
                       threshold: float) -> set:
    """Find near-duplicate rows within identical normalized titles."""

    drop = set()
    groups = frame.groupby("title_norm").indices

    for _, idx in tqdm(groups.items(), desc="near-dup", total=len(groups)):
        if len(idx) < 2 or len(idx) > 400:
            continue

        sub = unit_vectors[idx]
        sim = sub @ sub.T
        np.fill_diagonal(sim, 0.0)

        kept = []
        for local_i in range(len(idx)):
            if any(sim[local_i, k] >= threshold for k in kept):
                drop.add(int(idx[local_i]))
            else:
                kept.append(local_i)

    return drop


dup_idx = near_duplicate_ids(df, vectors_unit, CONFIG["near_dup_cosine"])
print(f"\nNear-duplicates found: {len(dup_idx):,}")

if dup_idx:
    keep_mask = ~df.index.isin(dup_idx)
    df = df[keep_mask].reset_index(drop=True)
    vectors = vectors[keep_mask]
    vectors_unit = vectors_unit[keep_mask]

print(f"After near-dup removal: {len(df):,}")

---
## 9. Location normalization

بدون `country` و `city` كأعمدة مستقلة، الفلترة الصارمة في Phase 4 مستحيلة.
القيم الخام غير متسقة: `Riyadh, SA` و `Dhahran, sa, Saudi Arabia`
و `Khamis, 7, Saudi Arabia`.

In [ ]:
COUNTRY_ALIASES = {
    "saudi arabia": "Saudi Arabia", "ksa": "Saudi Arabia",
    "sa": "Saudi Arabia", "sau": "Saudi Arabia",
    "kingdom of saudi arabia": "Saudi Arabia",
    "united arab emirates": "UAE", "uae": "UAE", "ae": "UAE",
    "qatar": "Qatar", "qa": "Qatar",
    "kuwait": "Kuwait", "kw": "Kuwait",
    "bahrain": "Bahrain", "bh": "Bahrain",
    "oman": "Oman", "om": "Oman",
    "egypt": "Egypt", "eg": "Egypt",
    "jordan": "Jordan", "jo": "Jordan",
    "united states": "United States", "usa": "United States",
    "us": "United States", "united kingdom": "United Kingdom",
    "uk": "United Kingdom", "india": "India", "in": "India",
    "pakistan": "Pakistan", "germany": "Germany", "france": "France",
    "canada": "Canada", "remote": None,
}

CITY_ALIASES = {
    "riyadh": ("Riyadh", "Riyadh Province"),
    "ar riyadh": ("Riyadh", "Riyadh Province"),
    "al riyadh": ("Riyadh", "Riyadh Province"),
    "jeddah": ("Jeddah", "Makkah Province"),
    "jiddah": ("Jeddah", "Makkah Province"),
    "makkah": ("Makkah", "Makkah Province"),
    "mecca": ("Makkah", "Makkah Province"),
    "madinah": ("Madinah", "Madinah Province"),
    "medina": ("Madinah", "Madinah Province"),
    "dammam": ("Dammam", "Eastern Province"),
    "dhahran": ("Dhahran", "Eastern Province"),
    "khobar": ("Al Khobar", "Eastern Province"),
    "al khobar": ("Al Khobar", "Eastern Province"),
    "jubail": ("Jubail", "Eastern Province"),
    "yanbu": ("Yanbu", "Madinah Province"),
    "tabuk": ("Tabuk", "Tabuk Province"),
    "abha": ("Abha", "Asir Province"),
    "khamis": ("Khamis Mushait", "Asir Province"),
    "khamis mushait": ("Khamis Mushait", "Asir Province"),
    "neom": ("NEOM", "Tabuk Province"),
    "qassim": ("Qassim", "Qassim Province"),
    "buraidah": ("Buraidah", "Qassim Province"),
    "hail": ("Hail", "Hail Province"),
    "jazan": ("Jazan", "Jazan Province"),
    "najran": ("Najran", "Najran Province"),
    "taif": ("Taif", "Makkah Province"),
    "dubai": ("Dubai", "Dubai"),
    "abu dhabi": ("Abu Dhabi", "Abu Dhabi"),
    "sharjah": ("Sharjah", "Sharjah"),
    "doha": ("Doha", "Doha"),
    "kuwait city": ("Kuwait City", "Al Asimah"),
    "manama": ("Manama", "Capital"),
    "muscat": ("Muscat", "Muscat"),
    "cairo": ("Cairo", "Cairo"),
}

REMOTE_RE = re.compile(r"\b(remote|work from home|wfh|anywhere)\b", re.I)
HYBRID_RE = re.compile(r"\bhybrid\b", re.I)


def parse_location(value: str):
    """Return (city, region, country, location_is_remote)."""

    if not value:
        return (None, None, None, False)

    is_remote = bool(REMOTE_RE.search(value))

    parts = [p.strip() for p in re.split(r"[,/|]", value) if p.strip()]
    parts = [p for p in parts if not re.fullmatch(r"\d+", p)]

    city = region = country = None

    for part in parts:
        key = part.lower().strip()

        if country is None and key in COUNTRY_ALIASES:
            mapped = COUNTRY_ALIASES[key]
            if mapped:
                country = mapped
            continue

        if city is None and key in CITY_ALIASES:
            city, region = CITY_ALIASES[key]
            continue

    # المدينة تكشف الدولة عندما لا تُذكر صراحة
    if country is None and city is not None:
        for key, (c, _) in CITY_ALIASES.items():
            if c == city:
                if key in ("dubai", "abu dhabi", "sharjah"):
                    country = "UAE"
                elif key == "doha":
                    country = "Qatar"
                elif key == "kuwait city":
                    country = "Kuwait"
                elif key == "manama":
                    country = "Bahrain"
                elif key == "muscat":
                    country = "Oman"
                elif key == "cairo":
                    country = "Egypt"
                else:
                    country = "Saudi Arabia"
                break

    if city is None and parts:
        candidate = REMOTE_RE.sub(" ", parts[0])
        candidate = re.sub(r"\s*[-–—]\s*", " ", candidate).strip()
        if candidate.lower() not in COUNTRY_ALIASES and len(candidate) > 2:
            city = candidate.title()

    return (city, region, country, is_remote)


parsed = df["location_raw"].map(parse_location)
df["city"] = parsed.map(lambda t: t[0])
df["region"] = parsed.map(lambda t: t[1])
df["country"] = parsed.map(lambda t: t[2])
df["location_is_remote"] = parsed.map(lambda t: t[3])

print("Country coverage:")
print(df["country"].value_counts(dropna=False).head(10))
print("\nCity coverage (top 12):")
print(df["city"].value_counts(dropna=False).head(12))

---
## 10. Dates and freshness

In [ ]:
now_utc = pd.Timestamp.now(tz="UTC")

for src, dst in [("seen_ms", "first_seen_at"), ("pub_ms", "published_at")]:
    if src in df.columns:
        df[dst] = pd.to_datetime(df[src], unit="ms", errors="coerce", utc=True)
    else:
        df[dst] = pd.NaT

df["posted_at"] = df["published_at"].fillna(df["first_seen_at"])
df["age_days"] = (now_utc - df["posted_at"]).dt.total_seconds() / 86400
df["age_days"] = df["age_days"].round(1)

print("Missing posted_at :", int(df["posted_at"].isna().sum()))
print("Oldest posting    :", df["posted_at"].min())
print("Newest posting    :", df["posted_at"].max())
print()
print(df["age_days"].describe().round(1).to_string())

if CONFIG["max_age_days"] is not None:
    before = len(df)
    keep = df["age_days"].isna() | (df["age_days"] <= CONFIG["max_age_days"])
    df, vectors, vectors_unit = df[keep], vectors[keep.values], vectors_unit[keep.values]
    df = df.reset_index(drop=True)
    print(f"Freshness filter: {before:,} -> {len(df):,}")

---
## 11. Quality gate

كل الفلاتر تُطبّق دفعة واحدة مع تقرير يوضّح سبب استبعاد كل صف.

In [ ]:
rules = {
    "empty_title": df["title_clean"].str.len() < CONFIG["min_title_chars"],
    "empty_jd": df["jd_clean"].str.len() == 0,
    "jd_too_short": df["jd_clean"].str.len() < CONFIG["min_jd_chars"],
    "jd_too_long": df["jd_clean"].str.len() > CONFIG["max_jd_chars"],
    "spam": df["is_spam"],
    "no_url": df["url_clean"].eq(""),
}

if CONFIG["allowed_languages"]:
    rules["language_excluded"] = ~df["jd_lang"].isin(CONFIG["allowed_languages"])

report = pd.DataFrame({
    "rule": list(rules.keys()),
    "rows_hit": [int(m.sum()) for m in rules.values()],
})
report["pct"] = (report["rows_hit"] / len(df) * 100).round(2)

print(f"Rows entering quality gate: {len(df):,}\n")
print(report.sort_values("rows_hit", ascending=False).to_string(index=False))

In [ ]:
drop_mask = np.zeros(len(df), dtype=bool)
for mask in rules.values():
    drop_mask |= mask.values

rejected_df = df[drop_mask].copy()
rejected_df.to_parquet(OUT / "rejected_jobs.parquet", index=False)

keep_mask = ~drop_mask
df = df[keep_mask].reset_index(drop=True)
vectors = vectors[keep_mask]
vectors_unit = vectors_unit[keep_mask]

print(f"Rejected : {len(rejected_df):,}  (saved to outputs/rejected_jobs.parquet)")
print(f"Retained : {len(df):,}")
assert len(df) == len(vectors), "dataframe and embedding matrix out of sync"

In [ ]:
# تأكيد نهائي على سلامة الطبقة النظيفة
checks = {
    "rows": len(df),
    "unique job_id": df["job_id"].nunique(),
    "duplicate content_hash": int(df["content_hash"].duplicated().sum()),
    "empty titles": int((df["title_clean"] == "").sum()),
    "empty JDs": int((df["jd_clean"] == "").sum()),
    "JDs below threshold": int((df["jd_chars"] < CONFIG["min_jd_chars"]).sum()),
    "missing country": int(df["country"].isna().sum()),
    "missing city": int(df["city"].isna().sum()),
    "spam remaining": int(df["is_spam"].sum()),
}

for key, value in checks.items():
    print(f"{key:<24}: {value:,}")

df[["job_id", "title_clean", "company_clean", "city", "country", "jd_words"]].head(10)

---
## 12. Structured feature schema (v2)

التعديلات الجوهرية مقارنة بالنسخة الأولى:

| المشكلة السابقة | الحل |
|---|---|
| `required_skills` يرجّع جُمل كاملة (`"Bachelor's degree in..."`) | فصل `qualifications` عن المهارات + فرض عبارات قصيرة ≤ 4 كلمات |
| قائمة مهارات واحدة تخلط التقني بالسلوكي باللغات | `hard_skills` / `soft_skills` / `tools_technologies` / `languages` |
| `work_arrangement` = Not Specified في 100% من العينة | السماح بالاستنتاج + علم `*_inferred` منفصل |
| تناقض: سنوات = 5 مع مستوى = Not Specified | قاعدة اتساق صريحة في التعليمات |
| لا راتب ولا متطلبات لغة/جنسية | حقول مخصصة |

علم `_inferred` هو المفتاح: يسمح لـ Phase 4 بالفلترة الصارمة على القيم المؤكدة،
ولـ Phase 5 بإعطاء وزن أقل للقيم المستنتجة.

In [ ]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field

ExperienceLevel = Literal[
    "Internship", "Entry Level", "Mid Level", "Senior Level",
    "Executive", "Not Specified",
]

EmploymentType = Literal[
    "Full-time", "Part-time", "Contract", "Internship",
    "Temporary", "Not Specified",
]

WorkArrangement = Literal["On-site", "Remote", "Hybrid", "Not Specified"]

EducationLevel = Literal[
    "High School", "Diploma", "Bachelor's Degree",
    "Master's Degree", "PhD", "Not Specified",
]

SalaryPeriod = Literal["hour", "day", "month", "year", "Not Specified"]


class JobFeatures(BaseModel):
    """Standardized structured representation of a job posting."""

    # ---------- Skills ----------
    hard_skills: List[str] = Field(
        description="Technical or domain skills. Short noun phrases, max 4 words."
    )
    soft_skills: List[str] = Field(
        description="Behavioural or interpersonal skills. Max 4 words each."
    )
    tools_technologies: List[str] = Field(
        description="Named tools, software, platforms, frameworks, standards."
    )
    languages: List[str] = Field(
        description="Spoken/written languages required, e.g. Arabic, English."
    )

    required_skills: List[str] = Field(
        description="Subset of the above that is explicitly mandatory."
    )
    preferred_skills: List[str] = Field(
        description="Subset that is optional, preferred or nice-to-have."
    )

    # ---------- Qualifications (NOT skills) ----------
    qualifications: List[str] = Field(
        description="Certifications, licences, degrees, memberships. Full phrases allowed."
    )

    # ---------- Experience ----------
    experience_level: ExperienceLevel
    experience_level_inferred: bool = Field(
        description="True if derived from wording rather than stated explicitly."
    )
    years_experience_min: Optional[float]
    years_experience_max: Optional[float]

    # ---------- Education ----------
    education_level: List[EducationLevel]
    education_field: List[str]

    # ---------- Employment ----------
    employment_type: EmploymentType
    employment_type_inferred: bool
    work_arrangement: WorkArrangement
    work_arrangement_inferred: bool

    # ---------- Compensation ----------
    salary_min: Optional[float]
    salary_max: Optional[float]
    salary_currency: Optional[str]
    salary_period: SalaryPeriod

    # ---------- Context ----------
    responsibilities: List[str]
    industry: Optional[str]
    department: Optional[str]
    nationality_requirement: Optional[str] = Field(
        description="e.g. 'Saudi nationals only'. Null if none stated."
    )

    # ---------- Self-assessment ----------
    extraction_confidence: float = Field(
        description="0.0-1.0 confidence that the description was informative enough."
    )


print("Schema fields:", len(JobFeatures.model_fields))

### 12.1 Extraction instructions

In [ ]:
SYSTEM_PROMPT = """
You are a job-posting information extraction system. You convert unstructured
job descriptions into a strict structured record.

GENERAL RULES

1. Never invent information. If it is not stated or clearly implied, use an
   empty list, null, or "Not Specified".
2. Work across ALL job domains — nursing, welding, teaching, accounting,
   logistics — not only technology.
3. Extract from the job description only. Ignore company boilerplate,
   EEO statements, and application instructions.

SKILLS — THE MOST IMPORTANT PART

4. A skill is a short noun phrase of AT MOST 4 words.
   GOOD: "financial reporting", "SQL", "patient care", "welding"
   BAD:  "Bachelor's degree in Administration or a relevant field"
   BAD:  "1 to 3 years of experience in similar roles"
   BAD:  "Ability to work independently under pressure in a team"
5. Degrees, years of experience, certifications and licences are NOT skills.
   They belong in `qualifications`, `education_level` or `years_experience_*`.
6. Split skills into four buckets:
   - hard_skills: technical/domain capabilities ("data modeling", "tax accounting")
   - soft_skills: behavioural ("teamwork", "negotiation")
   - tools_technologies: named products/standards ("Power BI", "SAP", "ISO 9001")
   - languages: spoken languages only ("Arabic", "English")
   A skill appears in exactly ONE bucket.
7. Use the canonical name of a tool: "Power BI" not "MS PowerBI" or "powerbi".
8. required_skills and preferred_skills must contain strings that already
   appear in one of the four buckets above, spelled identically.
   If the posting makes no distinction, put everything in required_skills
   and leave preferred_skills empty.

EXPERIENCE

9. years_experience_min / max come from explicit numbers only.
   "3+ years"     -> min 3, max null
   "3 to 5 years" -> min 3, max 5
   "at least 2"   -> min 2, max null
   No number      -> both null. Never assume 0 from "Entry Level".
10. Normalize: Junior/Graduate/Fresh Graduate -> Entry Level;
    Intermediate -> Mid Level; Senior/Lead/Principal/Staff -> Senior Level;
    Director/VP/Head/Chief -> Executive.
11. Consistency: if years_experience_min is known but no level is stated,
    INFER the level (0-1 Entry, 2-4 Mid, 5-9 Senior, 10+ Senior or Executive)
    and set experience_level_inferred = true.
    If the level is stated explicitly, set the flag to false.
    Never return "Not Specified" while a minimum number of years is present.

EMPLOYMENT TYPE AND WORK ARRANGEMENT

12. Return the category, never the raw phrase:
    "2-year contract" -> "Contract"; "full time position" -> "Full-time".
13. If the type is not stated but the posting clearly describes a permanent
    staff role, return "Full-time" with employment_type_inferred = true.
14. work_arrangement:
    - Explicit "remote"/"work from home" -> Remote, inferred = false
    - Explicit "hybrid" -> Hybrid, inferred = false
    - Explicit "on-site"/"in office" -> On-site, inferred = false
    - No statement, but a specific physical workplace, shift, site or
      facility is described -> On-site with work_arrangement_inferred = true
    - Genuinely unclear -> "Not Specified"
    Do NOT return "Not Specified" merely because the exact word is absent.

COMPENSATION

15. Extract salary numbers only if stated. Convert "10k" to 10000.
    salary_currency as an ISO-like code when identifiable: SAR, AED, USD, EUR.
    A range "8,000 - 12,000 SAR/month" -> min 8000, max 12000,
    currency SAR, period "month".

OTHER

16. education_level uses ONLY the allowed enum values. The field of study
    goes in education_field, one entry per field.
17. responsibilities: up to 8 concise bullet phrases describing the work.
18. nationality_requirement: fill only if the posting restricts by nationality
    or residency status.
19. extraction_confidence: 0.9+ for a detailed posting, 0.5 for a thin one,
    below 0.3 if the text is mostly marketing with no real job content.
"""

print(f"Prompt length: {len(SYSTEM_PROMPT):,} chars")

---
## 13. Async extraction pipeline

استبدال `.apply()` التسلسلي. الفروق العملية:

- **تزامن** عبر `asyncio` + semaphore — أسرع بعشرات المرات
- **cache دائم** بمفتاح `sha1(jd + model + prompt_version)` — إعادة التشغيل مجانية
- **retry** مع backoff أسّي و jitter، وتسجيل الفشل بدل ابتلاعه
- **checkpoint** كل N وظيفة

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from openai import AsyncOpenAI

try:
    from google.colab import userdata
    api_key = userdata.get("openai_key_new")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")

assert api_key, "OpenAI API key not found"
aclient = AsyncOpenAI(api_key=api_key)

print("Async client ready.")

In [ ]:
CACHE_PATH = CACHE / f"extraction_{CONFIG['prompt_version']}.jsonl"


def cache_key(jd: str) -> str:
    payload = f"{CONFIG['extraction_model']}|{CONFIG['prompt_version']}|{jd}"
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()


def load_cache() -> dict:
    store = {}
    if CACHE_PATH.exists():
        with CACHE_PATH.open(encoding="utf-8") as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                    store[rec["key"]] = rec["features"]
                except Exception:
                    continue
    return store


EXTRACTION_CACHE = load_cache()
print(f"Cached extractions: {len(EXTRACTION_CACHE):,}")

In [ ]:
def smart_truncate(jd: str, max_chars: int) -> str:
    """
    Truncate long JDs while preserving both ends.

    Naive head-truncation loses the requirements/skills section, which in
    many postings sits near the bottom (after company boilerplate). This
    keeps ~70% of the budget from the head and ~30% from the tail, joined
    with a marker, so both the role intro and the requirements survive.
    """

    if len(jd) <= max_chars:
        return jd

    head_budget = int(max_chars * 0.7)
    tail_budget = max_chars - head_budget - len("\n[... omitted ...]\n")

    return jd[:head_budget] + "\n[... omitted ...]\n" + jd[-tail_budget:]


_cache_fh = CACHE_PATH.open("a", encoding="utf-8")
_cache_lock = asyncio.Lock()

FAILURES = []


async def extract_one(job_id: str, jd: str, sem: asyncio.Semaphore) -> tuple:
    """Return (job_id, features_dict or None)."""

    jd = smart_truncate(jd, CONFIG["max_jd_chars_sent_to_llm"])
    key = cache_key(jd)

    if key in EXTRACTION_CACHE:
        return job_id, EXTRACTION_CACHE[key]

    delay = 2.0

    async with sem:
        for attempt in range(CONFIG["max_retries"]):
            try:
                response = await aclient.responses.parse(
                    model=CONFIG["extraction_model"],
                    input=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": jd},
                    ],
                    text_format=JobFeatures,
                )

                features = response.output_parsed.model_dump()

                async with _cache_lock:
                    EXTRACTION_CACHE[key] = features
                    _cache_fh.write(
                        json.dumps({"key": key, "job_id": job_id,
                                    "features": features}, ensure_ascii=False) + "\n"
                    )
                    _cache_fh.flush()

                return job_id, features

            except Exception as exc:
                is_last = attempt == CONFIG["max_retries"] - 1

                if is_last:
                    FAILURES.append({"job_id": job_id, "error": repr(exc)[:300]})
                    return job_id, None

                await asyncio.sleep(delay + np.random.rand())
                delay = min(delay * 2, 60)

    return job_id, None

In [ ]:
async def run_extraction(frame: pd.DataFrame) -> dict:
    sem = asyncio.Semaphore(CONFIG["concurrency"])

    tasks = [
        extract_one(row.job_id, row.jd_clean, sem)
        for row in frame.itertuples(index=False)
    ]

    results = {}
    done = 0

    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks),
                     desc="extracting"):
        job_id, features = await coro
        results[job_id] = features
        done += 1

        if done % CONFIG["checkpoint_every"] == 0:
            pd.DataFrame(
                [{"job_id": k, "features": json.dumps(v, ensure_ascii=False)}
                 for k, v in results.items() if v]
            ).to_parquet(WORK / "extraction_checkpoint.parquet", index=False)

    return results

### 13.1 Estimate tokens and cost before running

شغّل هذي الخلية دائمًا قبل الاستخراج. الرقم الثابت (`FIXED_TOKENS`) هو
حجم `SYSTEM_PROMPT` + الـ JSON schema، وهو نفسه في كل استدعاء — هذا بالضبط
ما تخصمه أغلب مزوّدي الـ API عبر prompt caching بنسبة تصل ~90% (`CACHE_DISCOUNT`
أدناه)، بشرط ألا يتغير حرف واحد من الـ system prompt بين الاستدعاءات.

عدّل `PRICE_IN_PER_M` و `PRICE_OUT_PER_M` بسعر الموديل اللي تستخدمه لتشوف
التكلفة التقريبية بالدولار قبل الالتزام.

In [ ]:
FIXED_TOKENS = round((len(SYSTEM_PROMPT) + len(JobFeatures.model_json_schema().__repr__())) / 4)
OUTPUT_TOKENS_PER_JOB = 600          # تقدير متوسط لحجم الـ JSON الناتج
CACHE_DISCOUNT = 0.90                # خصم توكنز البرومبت الثابت عبر caching (إن وُجد)

PRICE_IN_PER_M = 0.0                 # $ لكل مليون توكن إدخال — عدّله لسعر موديلك
PRICE_OUT_PER_M = 0.0                # $ لكل مليون توكن إخراج

target_df = df
if CONFIG["max_jobs_to_extract"]:
    target_df = df.head(CONFIG["max_jobs_to_extract"])

already_cached = sum(
    cache_key(smart_truncate(j, CONFIG["max_jd_chars_sent_to_llm"])) in EXTRACTION_CACHE
    for j in target_df["jd_clean"]
)
to_process = len(target_df) - already_cached

avg_jd_tokens = (
    target_df["jd_clean"].str.slice(0, CONFIG["max_jd_chars_sent_to_llm"])
    .str.len().mean() / 4
)

n_truncated = (target_df["jd_clean"].str.len() > CONFIG["max_jd_chars_sent_to_llm"]).sum()
pct_truncated = n_truncated / len(target_df) * 100

input_tokens_no_cache = to_process * (FIXED_TOKENS + avg_jd_tokens)
input_tokens_cached = to_process * (FIXED_TOKENS * (1 - CACHE_DISCOUNT) + avg_jd_tokens)
output_tokens = to_process * OUTPUT_TOKENS_PER_JOB

cost_no_cache = (input_tokens_no_cache / 1e6 * PRICE_IN_PER_M
                 + output_tokens / 1e6 * PRICE_OUT_PER_M)
cost_cached = (input_tokens_cached / 1e6 * PRICE_IN_PER_M
               + output_tokens / 1e6 * PRICE_OUT_PER_M)

print(f"Jobs total          : {len(target_df):,}")
print(f"Already cached (ours): {already_cached:,}")
print(f"To process           : {to_process:,}")
print(f"Fixed tokens/call    : ~{FIXED_TOKENS:,}  (system prompt + schema)")
print(f"Avg JD tokens/call   : ~{avg_jd_tokens:,.0f}")
print(f"Jobs truncated       : {n_truncated:,} ({pct_truncated:.1f}%)  "
      f"[head+tail smart_truncate — requirements section preserved]")
print()
print(f"Input tokens  (no prompt-cache) : {input_tokens_no_cache/1e6:.2f}M")
print(f"Input tokens  (with prompt-cache): {input_tokens_cached/1e6:.2f}M")
print(f"Output tokens                    : {output_tokens/1e6:.2f}M")
print()
if PRICE_IN_PER_M or PRICE_OUT_PER_M:
    print(f"Est. cost (no cache) : ${cost_no_cache:,.2f}")
    print(f"Est. cost (w/ cache) : ${cost_cached:,.2f}")
else:
    print("Set PRICE_IN_PER_M / PRICE_OUT_PER_M above to see cost in $.")

> **قبل تشغيل الخلية التالية:** إذا كان `to_process` كبيرًا ولا ترغب
> بتشغيل الاستخراج كامل الآن، اضبط `CONFIG["max_jobs_to_extract"] = 200`
> في خلية الإعدادات (القسم 2) وأعد التشغيل من هناك — جرّب على عيّنة صغيرة
> أولًا، راجع جودة الاستخراج في القسم 15، ثم ارفع الرقم أو احذفه (`None`)
> لتشغيله على الكل. أي تشغيل جزئي محفوظ في الـ cache ولن يُعاد دفعه.

In [ ]:
target_df = df
if CONFIG["max_jobs_to_extract"]:
    target_df = df.head(CONFIG["max_jobs_to_extract"])

print(f"Jobs to extract : {len(target_df):,}")
print(f"Already cached  : "
      f"{sum(cache_key(smart_truncate(j, CONFIG['max_jd_chars_sent_to_llm'])) in EXTRACTION_CACHE for j in target_df['jd_clean']):,}")

start = time.time()
results = asyncio.get_event_loop().run_until_complete(run_extraction(target_df))
elapsed = time.time() - start

ok = sum(1 for v in results.values() if v)
print(f"\nSucceeded : {ok:,}")
print(f"Failed    : {len(FAILURES):,}")
print(f"Elapsed   : {elapsed/60:.1f} min")

if FAILURES:
    display(pd.DataFrame(FAILURES).head(10))

### 13.1 Expand features into columns

In [ ]:
FEATURE_FIELDS = list(JobFeatures.model_fields.keys())

LIST_FIELDS = [
    f for f in FEATURE_FIELDS
    if str(JobFeatures.model_fields[f].annotation).startswith("typing.List")
]

feat_rows = []
for job_id in df["job_id"]:
    features = results.get(job_id)
    if features is None:
        features = {f: ([] if f in LIST_FIELDS else None) for f in FEATURE_FIELDS}
        features["extraction_status"] = "failed"
    else:
        features = dict(features)
        features["extraction_status"] = "ok"
    features["job_id"] = job_id
    feat_rows.append(features)

feat_df = pd.DataFrame(feat_rows)

prepared_df = df.merge(feat_df, on="job_id", how="left", validate="one_to_one")

# قوائم فارغة بدل None حتى لا تنكسر عمليات parquet والمطابقة
for field in LIST_FIELDS:
    prepared_df[field] = prepared_df[field].map(
        lambda v: list(v) if isinstance(v, (list, np.ndarray)) else []
    )

print(f"Rows: {len(prepared_df):,} | Columns: {prepared_df.shape[1]}")
print(prepared_df["extraction_status"].value_counts().to_dict())

---
## 14. Skill normalization

بدون هذه الخطوة تفشل المطابقة في Phase 5: `Power BI` و `PowerBI` و `MS Power BI`
ثلاث مهارات مختلفة عند مقارنة النصوص.

In [ ]:
SKILL_ALIASES = {
    # data / bi
    "powerbi": "power bi", "ms power bi": "power bi",
    "microsoft power bi": "power bi",
    "ms excel": "excel", "microsoft excel": "excel",
    "advanced excel": "excel", "ms office": "microsoft office",
    "office suite": "microsoft office", "msoffice": "microsoft office",
    "sql server": "microsoft sql server", "mssql": "microsoft sql server",
    "postgres": "postgresql", "ms sql": "microsoft sql server",
    "google data studio": "looker studio",
    "data visualisation": "data visualization",
    "data analytics": "data analysis",
    "statistical analysis": "statistics",
    "machine learning (ml)": "machine learning",
    "ml": "machine learning", "ai": "artificial intelligence",
    "nlp": "natural language processing",
    "etl pipelines": "etl", "etl processes": "etl",
    # engineering / software
    "js": "javascript", "reactjs": "react", "react.js": "react",
    "nodejs": "node.js", "node js": "node.js",
    "py": "python", "python3": "python",
    "c sharp": "c#", "golang": "go",
    "rest apis": "rest api", "restful api": "rest api",
    "ci cd": "ci/cd", "cicd": "ci/cd",
    "aws cloud": "aws", "amazon web services": "aws",
    "microsoft azure": "azure", "gcp": "google cloud",
    "k8s": "kubernetes",
    # business
    "ms project": "microsoft project",
    "erp systems": "erp", "sap erp": "sap",
    "kpi reporting": "kpi reporting",
    "stakeholder management": "stakeholder management",
    "project mgmt": "project management",
    # soft
    "communication skills": "communication",
    "verbal communication": "communication",
    "written communication": "written communication",
    "team work": "teamwork", "team player": "teamwork",
    "problem-solving": "problem solving",
    "time-management": "time management",
    "attention to detail": "attention to detail",
    "interpersonal skills": "interpersonal skills",
    "leadership skills": "leadership",
    "analytical skills": "analytical thinking",
    "analytical thinking skills": "analytical thinking",
    # languages
    "english language": "english", "arabic language": "arabic",
    "fluent english": "english", "native arabic": "arabic",
    "english (fluent)": "english",
}

ACRONYM_KEEP = {"sql", "aws", "gcp", "erp", "sap", "api", "etl", "bi", "qa",
                "hr", "ui", "ux", "iso", "css", "html", "php", "ios", "crm",
                "kpi", "cad", "plc", "hvac", "ccna", "pmp", "cpa", "acca"}

NOISE_RE = re.compile(
    r"^(ability to|able to|experience (in|with)|knowledge of|proficiency in|"
    r"strong |excellent |good |demonstrated |proven |solid |working )",
    re.I,
)


def normalize_skill(skill: str) -> str:
    """Map a raw skill string to its canonical form, or '' if unusable."""

    if not isinstance(skill, str):
        return ""

    text = clean_text(skill, keep_newlines=False).lower()
    text = NOISE_RE.sub("", text)
    text = re.sub(r"[\(\)\[\]\.,;:]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip(" -–—/")

    if not text or len(text) < 2:
        return ""

    # الجُمل الطويلة متطلبات لا مهارات
    if len(text.split()) > 5:
        return ""

    text = SKILL_ALIASES.get(text, text)

    if text.endswith("s") and text[:-1] in SKILL_ALIASES.values():
        text = text[:-1]

    return SKILL_ALIASES.get(text, text)


def display_skill(canonical: str) -> str:
    if canonical in ACRONYM_KEEP:
        return canonical.upper()
    return " ".join(
        w.upper() if w in ACRONYM_KEEP else w.capitalize()
        for w in canonical.split()
    )


def normalize_skill_list(values) -> list:
    seen, out = set(), []
    for item in (values or []):
        canon = normalize_skill(item)
        if canon and canon not in seen:
            seen.add(canon)
            out.append(canon)
    return out

In [ ]:
for field in ["hard_skills", "soft_skills", "tools_technologies",
              "languages", "required_skills", "preferred_skills"]:
    prepared_df[f"{field}_norm"] = prepared_df[field].map(normalize_skill_list)

# مجموعة موحّدة تُستخدم في المطابقة
prepared_df["all_skills_norm"] = prepared_df.apply(
    lambda r: sorted(set(
        r["hard_skills_norm"] + r["tools_technologies_norm"] + r["soft_skills_norm"]
    )),
    axis=1,
)

prepared_df["n_skills"] = prepared_df["all_skills_norm"].map(len)

print("Skills per job:")
print(prepared_df["n_skills"].describe().round(1))

In [ ]:
from collections import Counter

counter = Counter()
for skills in prepared_df["all_skills_norm"]:
    counter.update(skills)

vocab_df = (
    pd.DataFrame(counter.most_common(), columns=["skill_canonical", "job_count"])
    .assign(
        display_name=lambda d: d["skill_canonical"].map(display_skill),
        doc_freq=lambda d: (d["job_count"] / len(prepared_df)).round(5),
    )
)

# IDF جاهز لترجيح المهارات النادرة في Phase 5
vocab_df["idf"] = np.log(len(prepared_df) / (1 + vocab_df["job_count"])).round(4)

print(f"Vocabulary size: {len(vocab_df):,}")
vocab_df.head(25)

---
## 15. Extraction quality assurance

In [ ]:
ok_df = prepared_df[prepared_df["extraction_status"] == "ok"]

print(f"Successful extractions: {len(ok_df):,} / {len(prepared_df):,}\n")

coverage = pd.DataFrame([
    {"field": "hard_skills", "filled_pct": (ok_df["hard_skills_norm"].map(len) > 0).mean()},
    {"field": "tools", "filled_pct": (ok_df["tools_technologies_norm"].map(len) > 0).mean()},
    {"field": "experience_level", "filled_pct": (ok_df["experience_level"] != "Not Specified").mean()},
    {"field": "years_experience_min", "filled_pct": ok_df["years_experience_min"].notna().mean()},
    {"field": "education_level", "filled_pct": (ok_df["education_level"].map(len) > 0).mean()},
    {"field": "employment_type", "filled_pct": (ok_df["employment_type"] != "Not Specified").mean()},
    {"field": "work_arrangement", "filled_pct": (ok_df["work_arrangement"] != "Not Specified").mean()},
    {"field": "salary", "filled_pct": ok_df["salary_min"].notna().mean()},
    {"field": "industry", "filled_pct": ok_df["industry"].notna().mean()},
])
coverage["filled_pct"] = (coverage["filled_pct"] * 100).round(1)

print(coverage.to_string(index=False))

In [ ]:
# نسبة القيم المستنتجة مقابل المصرّح بها — مهمة لضبط صرامة Phase 4
for field in ["experience_level", "employment_type", "work_arrangement"]:
    flag = f"{field}_inferred"
    stated = int((~ok_df[flag].astype(bool)).sum())
    inferred = int(ok_df[flag].astype(bool).sum())
    print(f"{field:<20} stated={stated:>6,}  inferred={inferred:>6,}")

print()
for field in ["experience_level", "employment_type", "work_arrangement"]:
    print(f"--- {field} ---")
    print(ok_df[field].value_counts().to_string())
    print()

In [ ]:
# فحص هلوسة تقريبي: هل المهارة المستخرجة موجودة فعلًا في النص؟
def grounding_ratio(row) -> float:
    skills = row["hard_skills_norm"] + row["tools_technologies_norm"]
    if not skills:
        return np.nan

    jd_lower = row["jd_flat"].lower()
    hits = sum(1 for s in skills if s.split()[0] in jd_lower)
    return hits / len(skills)


sample = ok_df.sample(n=min(400, len(ok_df)), random_state=42).copy()
sample["grounding"] = sample.apply(grounding_ratio, axis=1)

print("Grounding ratio (extracted skill token present in JD):")
print(sample["grounding"].describe().round(3))

weak = sample[sample["grounding"] < 0.5]
print(f"\nJobs below 0.5 grounding: {len(weak)} / {len(sample)}")
weak[["title_clean", "hard_skills_norm", "grounding"]].head(5)

In [ ]:
# مراجعة يدوية: 3 وظائف كاملة
for _, row in ok_df.sample(n=min(3, len(ok_df)), random_state=7).iterrows():
    print("=" * 100)
    print("TITLE      :", row["title_clean"])
    print("COMPANY    :", row["company_clean"])
    print("LOCATION   :", row["city"], "|", row["country"])
    print("CONFIDENCE :", row["extraction_confidence"])
    print("\nJD (first 700 chars):")
    print(row["jd_clean"][:700])
    print("\nEXTRACTED")
    print("  hard_skills :", row["hard_skills_norm"][:12])
    print("  tools       :", row["tools_technologies_norm"][:12])
    print("  soft_skills :", row["soft_skills_norm"][:8])
    print("  languages   :", row["languages_norm"])
    print("  quals       :", row["qualifications"][:4])
    print("  experience  :", row["experience_level"],
          f"(inferred={row['experience_level_inferred']})",
          f"years={row['years_experience_min']}-{row['years_experience_max']}")
    print("  education   :", row["education_level"], row["education_field"])
    print("  employment  :", row["employment_type"], "|", row["work_arrangement"])
    print("  salary      :", row["salary_min"], row["salary_max"],
          row["salary_currency"], row["salary_period"])
    print()

---
## 16. Save Phase 1 outputs

In [ ]:
FINAL_COLUMNS = [
    # identity
    "job_id", "ats", "slug", "id", "url_clean",
    # core text
    "title_clean", "company_clean", "jd_clean", "jd_flat",
    "jd_chars", "jd_words", "jd_lang",
    # location
    "location_raw", "city", "region", "country", "location_is_remote",
    # time
    "first_seen_at", "published_at", "posted_at", "age_days",
    # raw extracted
    "hard_skills", "soft_skills", "tools_technologies", "languages",
    "required_skills", "preferred_skills", "qualifications",
    # normalized
    "hard_skills_norm", "soft_skills_norm", "tools_technologies_norm",
    "languages_norm", "required_skills_norm", "preferred_skills_norm",
    "all_skills_norm", "n_skills",
    # structured
    "experience_level", "experience_level_inferred",
    "years_experience_min", "years_experience_max",
    "education_level", "education_field",
    "employment_type", "employment_type_inferred",
    "work_arrangement", "work_arrangement_inferred",
    "salary_min", "salary_max", "salary_currency", "salary_period",
    "responsibilities", "industry", "department", "nationality_requirement",
    # meta
    "extraction_status", "extraction_confidence", "content_hash",
]

final_df = prepared_df[[c for c in FINAL_COLUMNS if c in prepared_df.columns]].copy()

jobs_path = OUT / "jobs_prepared.parquet"
final_df.to_parquet(jobs_path, index=False, compression="snappy")

print(f"Saved {jobs_path}  ({len(final_df):,} rows, {final_df.shape[1]} cols, "
      f"{jobs_path.stat().st_size/1e6:.1f} MB)")

In [ ]:
# الـ embeddings تُحفظ منفصلة — Phase 3 يحمّلها مباشرة بدون تحميل النصوص
np.save(OUT / "job_embeddings.npy", vectors_unit.astype(np.float32))
np.save(OUT / "job_ids.npy", final_df["job_id"].to_numpy())

vocab_df.to_parquet(OUT / "skill_vocabulary.parquet", index=False)

meta = {
    "run_id": RUN_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "n_jobs": int(len(final_df)),
    "n_rejected": int(len(rejected_df)),
    "embedding_dim": int(vectors_unit.shape[1]),
    "embedding_model": "text-embedding-3-small",
    "extraction_model": CONFIG["extraction_model"],
    "prompt_version": CONFIG["prompt_version"],
    "extraction_failures": len(FAILURES),
    "vocabulary_size": int(len(vocab_df)),
    "config": CONFIG,
}

(OUT / "run_metadata.json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(json.dumps({k: v for k, v in meta.items() if k != "config"}, indent=2))

In [ ]:
DATA_DICTIONARY = """# jobs_prepared.parquet — Data Dictionary

One row per unique job posting. Produced by PHASE 1 and consumed by PHASE 3-5.

## Identity
| Column | Type | Notes |
|---|---|---|
| job_id | str | sha1(ats|slug|id)[:16]. Stable across runs. Join key everywhere. |
| ats / slug / id | str | Source system identifiers. |
| url_clean | str | Canonical application URL. |
| content_hash | str | Fingerprint used for deduplication. |

## Text
| Column | Type | Notes |
|---|---|---|
| title_clean | str | Whitespace-normalized title. |
| company_clean | str | Normalized employer name. |
| jd_clean | str | HTML-decoded description, **line structure preserved**. Use for LLM and display. |
| jd_flat | str | Single-line version. Use for keyword search and similarity. |
| jd_lang | str | en / ar / mixed / unknown. |

## Location
| Column | Type | Notes |
|---|---|---|
| city, region, country | str/null | Parsed. `country` drives hard filtering in PHASE 4. |
| location_is_remote | bool | Remote keyword present in the location string. |

## Time
| Column | Type | Notes |
|---|---|---|
| posted_at | datetime UTC | published_at, falling back to first_seen_at. |
| age_days | float | Days since posted_at at build time. |

## Skills
Raw fields (`hard_skills`, `soft_skills`, `tools_technologies`, `languages`,
`required_skills`, `preferred_skills`) hold the LLM output verbatim.
The `*_norm` variants are canonicalized via the alias map and are the ones
matching should use. `all_skills_norm` is the deduplicated union of
hard skills, tools and soft skills.

## Structured fields
| Column | Notes |
|---|---|
| experience_level | Internship / Entry / Mid / Senior / Executive / Not Specified |
| experience_level_inferred | True = derived, not stated. Weight lower in scoring. |
| years_experience_min/max | Explicit numbers only; null when unstated. |
| education_level | Controlled enum list. |
| employment_type, work_arrangement | Each paired with an `_inferred` flag. |
| salary_min/max/currency/period | Null when not disclosed (the common case). |
| nationality_requirement | e.g. "Saudi nationals only". Null if unrestricted. |
| extraction_confidence | 0-1 self-assessment. Filter below 0.3 for strict use. |

## Companion files
| File | Contents |
|---|---|
| job_embeddings.npy | float32 (n, 1536), L2-normalized, row order == job_ids.npy |
| job_ids.npy | job_id array aligning embeddings to the parquet |
| skill_vocabulary.parquet | canonical skill, job_count, doc_freq, idf |
| rejected_jobs.parquet | Rows removed by the quality gate, with reason columns |
| run_metadata.json | Full config and counts for reproducibility |

## PHASE 4 filtering contract
Hard filters should use: `country`, `city`, `work_arrangement`
(when `work_arrangement_inferred == False`), `years_experience_min`,
`education_level`, `nationality_requirement`, `age_days`.
Never hard-filter on an inferred value.
"""

(OUT / "DATA_DICTIONARY.md").write_text(DATA_DICTIONARY, encoding="utf-8")
print("Data dictionary written.")

In [ ]:
# التحقق النهائي: أعد التحميل وتأكد من التوافق
reloaded = pd.read_parquet(OUT / "jobs_prepared.parquet")
emb = np.load(OUT / "job_embeddings.npy")
ids = np.load(OUT / "job_ids.npy", allow_pickle=True)

assert len(reloaded) == len(emb) == len(ids), "row count mismatch"
assert (reloaded["job_id"].to_numpy() == ids).all(), "embedding order mismatch"
assert reloaded["job_id"].is_unique, "duplicate job_id"
assert np.allclose(np.linalg.norm(emb, axis=1), 1.0, atol=1e-3), "vectors not unit-norm"

print("PHASE 1 COMPLETE")
print(f"  jobs            : {len(reloaded):,}")
print(f"  columns         : {reloaded.shape[1]}")
print(f"  embeddings      : {emb.shape}")
print(f"  skill vocabulary: {len(vocab_df):,}")
print(f"  extraction ok   : {(reloaded['extraction_status'] == 'ok').mean():.1%}")
print("\nReady for PHASE 2 (Candidate Profile) and PHASE 3 (Retrieval).")

---
## 17. Persist outputs (Colab storage is temporary!)

مجلد `/content/` في Colab يُمسح عند انتهاء الجلسة. شغّل إحدى الطريقتين
(أو كلاهما) **فورًا** بعد نجاح القسم 16، قبل ما تسكّر المتصفح.

In [ ]:
# الطريقة 1: حفظ مباشر على Google Drive (الأنسب للاستمرار لاحقًا)
from google.colab import drive
drive.mount("/content/drive")

DRIVE_OUT = Path("/content/drive/MyDrive/ai_job_agent/phase1_outputs")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

import shutil
for f in OUT.iterdir():
    shutil.copy2(f, DRIVE_OUT / f.name)

print(f"Copied {len(list(OUT.iterdir()))} files to:")
print(f"  {DRIVE_OUT}")
print("\nFiles:")
for f in sorted(DRIVE_OUT.iterdir()):
    print(f"  {f.name:<30} {f.stat().st_size/1e6:.2f} MB")

In [ ]:
# الطريقة 2: تنزيل مباشر كملف zip على جهازك
import shutil as _shutil
from google.colab import files as _colab_files

zip_path = _shutil.make_archive("phase1_outputs", "zip", OUT)
print(f"Created {zip_path} ({Path(zip_path).stat().st_size/1e6:.1f} MB)")

_colab_files.download(zip_path)

---
## 18. Auto-upload to GitHub

يرفع الملفات الستة تلقائيًا لمستودعك على GitHub. **قبل التشغيل:**

1. أنشئ Personal Access Token (صلاحية `repo` فقط) من
   GitHub → Settings → Developer settings → Personal access tokens
2. في Colab: أيقونة المفتاح 🔑 يسار الشاشة → Add new secret →
   الاسم `github_token`، القيمة التوكن → فعّل الوصول لهذا النوتبوك
3. عدّل `github_repo` في القسم 2 (`CONFIG`) لاسم مستودعك، مثلًا
   `"khaled/ai-job-agent-data"`. المستودع لازم يكون موجودًا مسبقًا على GitHub.

الخلية تعمل commit فقط إذا كان هناك تغيير فعلي — تشغيلها مرتين بدون
تعديل الملفات لن ينشئ commit فارغ.

In [ ]:
from google.colab import userdata

try:
    GH_TOKEN = userdata.get("github_token")
except Exception:
    GH_TOKEN = None

assert GH_TOKEN, (
    "لم يتم العثور على github_token في Colab Secrets. "
    "أضفه من أيقونة المفتاح 🔑 يسار الشاشة قبل تشغيل هذه الخلية."
)
assert "/" in CONFIG["github_repo"] and "USERNAME" not in CONFIG["github_repo"], (
    "عدّل CONFIG['github_repo'] في القسم 2 إلى 'اسم_المستخدم/اسم_المستودع' الفعلي."
)

print(f"Target repo   : {CONFIG['github_repo']}")
print(f"Target branch : {CONFIG['github_branch']}")
print(f"Target folder : {CONFIG['github_subdir']}")

In [ ]:
import subprocess

REPO_DIR = Path("/content/gh_repo")
REPO_URL = f"https://{GH_TOKEN}@github.com/{CONFIG['github_repo']}.git"


def run(cmd: list, cwd=None, check=True):
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    # لا نطبع stdout/stderr حرفيًا لأنها قد تحتوي التوكن ضمن remote url
    safe_out = result.stdout.replace(GH_TOKEN, "***")
    safe_err = result.stderr.replace(GH_TOKEN, "***")
    if safe_out.strip():
        print(safe_out.strip())
    if check and result.returncode != 0:
        print(safe_err.strip())
        raise RuntimeError(f"Command failed: {' '.join(cmd[:2])}...")
    return result


run(["git", "config", "--global", "user.email", "colab@ai-job-agent.local"])
run(["git", "config", "--global", "user.name", "AI Job Agent Pipeline"])

if REPO_DIR.exists():
    print("Repo already cloned — pulling latest changes...")
    run(["git", "pull", "origin", CONFIG["github_branch"]], cwd=REPO_DIR)
else:
    run(["git", "clone", "--branch", CONFIG["github_branch"],
         REPO_URL, str(REPO_DIR)])

print("Repo ready at", REPO_DIR)

In [ ]:
target_dir = REPO_DIR / CONFIG["github_subdir"]
target_dir.mkdir(parents=True, exist_ok=True)

copied = []
for f in OUT.iterdir():
    if f.is_file():
        shutil.copy2(f, target_dir / f.name)
        copied.append(f.name)

print(f"Copied {len(copied)} files to {target_dir}:")
for name in sorted(copied):
    size_mb = (target_dir / name).stat().st_size / 1e6
    print(f"  {name:<30} {size_mb:>7.2f} MB")

# أي ملف أكبر من 50MB يُتابع عبر Git LFS تلقائيًا بدل الرفع العادي
large_files = [n for n in copied if (target_dir / n).stat().st_size > 50_000_000]
if large_files:
    print(f"\nLarge files detected (>50MB), enabling Git LFS: {large_files}")
    run(["git", "lfs", "install"], cwd=REPO_DIR, check=False)
    for pattern in {f"*{Path(n).suffix}" for n in large_files}:
        run(["git", "lfs", "track", pattern], cwd=REPO_DIR, check=False)
    run(["git", "add", ".gitattributes"], cwd=REPO_DIR, check=False)

In [ ]:
run(["git", "add", "."], cwd=REPO_DIR)

status = run(["git", "status", "--porcelain"], cwd=REPO_DIR)

if not status.stdout.strip():
    print("No changes to commit — files already up to date on GitHub.")
else:
    commit_msg = f"Phase 1 outputs — run {RUN_ID} ({len(reloaded):,} jobs)"
    run(["git", "commit", "-m", commit_msg], cwd=REPO_DIR)
    run(["git", "push", "origin", CONFIG["github_branch"]], cwd=REPO_DIR)

    repo_url = f"https://github.com/{CONFIG['github_repo']}/tree/{CONFIG['github_branch']}/{CONFIG['github_subdir']}"
    print(f"\nPushed successfully.")
    print(f"View files: {repo_url}")

---
## ما الذي تغيّر مقارنة بالنسخة الأولى

| # | التعديل | الأثر |
|---|---|---|
| 1 | حذف تفضيلات المستخدم و `sim` من Phase 1 | المخرج أصبح corpus عامًا لا نتيجة بحث |
| 2 | نقاط ارتكاز متعددة بدل واحدة | تغطية أوسع للمجالات |
| 3 | `job_id` ثابت | dedup، cache، وربط نتائج Phase 9 |
| 4 | فك HTML + الحفاظ على الأسطر | اختفاء `&amp;` وتحسّن جودة الاستخراج |
| 5 | كشف اللغة | فصل الإعلانات العربية عن الإنجليزية |
| 6 | فلترة spam مع تقرير المرفوضات | إزالة إعلانات التجميع من الـ retrieval |
| 7 | dedup ثلاثي الطبقات بالـ embeddings | كشف نفس الوظيفة عبر وكالات مختلفة |
| 8 | تطبيع الموقع إلى city/region/country | Phase 4 صار ممكنًا |
| 9 | فصل `qualifications` عن المهارات + 4 دلاء | مطابقة Phase 5 صارت ذات معنى |
| 10 | أعلام `_inferred` | فلترة صارمة على المؤكد فقط |
| 11 | async + cache + retry | من ساعات إلى دقائق، وإعادة التشغيل مجانية |
| 12 | قاموس مهارات + IDF | ترجيح المهارات النادرة في التقييم |
| 13 | حفظ الـ embeddings منفصلة | Phase 3 لا يحتاج تحميل النصوص |

## الخطوة التالية — PHASE 2

نوتبوك منفصل يبني `candidate_profile` بنفس هذا الـ schema بالضبط:
نفس دلاء المهارات، نفس التطبيع، نفس قيم enum. التماثل هذا هو ما يجعل
المطابقة في Phase 5 مجرد مقارنة حقل بحقل بدل معالجة نصية إضافية.